In [26]:
! pip install kagglehub

In [27]:
import pandas as pd 
import numpy as np
import kagglehub
import os

In [28]:
# Descargar la última version del dataset de Kaggle
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

# Revisar que archivos contiene
print(os.listdir(path))

Path to dataset files: C:\Users\Ivanna\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Comprensión de los datos

In [29]:
import pandas as pd
import os

customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv"))
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(
    os.path.join(path, "product_category_name_translation.csv")
)

## Dimensión de las bases de datos

In [30]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

Customers: 99441 filas, 5 columnas
Geolocation: 1000163 filas, 5 columnas
Orders: 99441 filas, 8 columnas
Order Items: 112650 filas, 7 columnas
Order Payments: 103886 filas, 5 columnas
Order Reviews: 99224 filas, 7 columnas
Products: 32951 filas, 9 columnas
Sellers: 3095 filas, 4 columnas
Category Translation: 71 filas, 2 columnas


## Duplicados

In [31]:
for nombre, df in datasets.items():
    print(f"{nombre}: {df.duplicated().sum()} duplicados")

Customers: 0 duplicados
Geolocation: 261831 duplicados
Orders: 0 duplicados
Order Items: 0 duplicados
Order Payments: 0 duplicados
Order Reviews: 0 duplicados
Products: 0 duplicados
Sellers: 0 duplicados
Category Translation: 0 duplicados


En este caso estuvimos viendo los duplicados en las base de datos en dónde pudimos encontrar que la única con estos fué Geolocation con 261831 duplicados, pero estos se estarán viendo más adelante en la parte de la exploración de los datos. 

## Verificar las llaves primarias

In [32]:
print(customers["customer_id"].is_unique)
print(orders["order_id"].is_unique)
print(products["product_id"].is_unique)
print(sellers["seller_id"].is_unique)
print(category_translation["product_category_name"].is_unique)

True
True
True
True
True


Cómo podemos ver las llaves primarias de las bases de datos son las siguientes:

* customer_id para la base de datos Customers
* order_id para la base de datos Orders
* product_id para la base de datos Products
* geolocation_id para la base de datos Geolocation. 

La cuales nos ayudarán a conectar las bases de datos de acuerdo a nuestro objetivo de negocio y poder llegar a una solución.

## Revisar los tipos de datos

In [33]:
for nombre, df in datasets.items():
    print(f"\n{nombre}")
    print(df.dtypes)


Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Order Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype:

Se verificaron los tipos de datos de las nueve tablas del conjunto de datos. Los identificadores se encuentran almacenados como variables de tipo object, mientras que las variables numéricas presentan tipos int64 y float64, lo cual es consistente con la naturaleza de la variable. Se identificó que las variables correspondientes a fechas y horas en las tablas orders, order_items y order_reviews se encuentran almacenadas como object; por tanto, durante la fase de preparación de los datos se convertirán al tipo datetime para facilitar el análisis temporal. No se identificaron inconsistencias relevantes en los demás tipos de datos.

## Completitud de los datos

### Calcular valores nulos

In [34]:
def completitud(df):
    reporte = pd.DataFrame({
        "Valores no nulos": df.notnull().sum(),
        "Valores nulos": df.isnull().sum(),
        "Porcentaje de nulos (%)": round(df.isnull().mean() * 100, 2)
    })

    return reporte.sort_values("Porcentaje de nulos (%)", ascending=False)

In [36]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"\n===== {nombre} =====")
    print(completitud(df))


===== Customers =====
                          Valores no nulos  Valores nulos  \
customer_id                          99441              0   
customer_unique_id                   99441              0   
customer_zip_code_prefix             99441              0   
customer_city                        99441              0   
customer_state                       99441              0   

                          Porcentaje de nulos (%)  
customer_id                                   0.0  
customer_unique_id                            0.0  
customer_zip_code_prefix                      0.0  
customer_city                                 0.0  
customer_state                                0.0  

===== Geolocation =====
                             Valores no nulos  Valores nulos  \
geolocation_zip_code_prefix           1000163              0   
geolocation_lat                       1000163              0   
geolocation_lng                       1000163              0   
geolocation_city 

En este caso evaluamos la completitud de las nueve tablas del conjunto de datos mediante la cantidad de valores nulos por variable.

Los principales hallazgos que pudimos encontrar fueron los siguientes:

* Customers: presenta un 100 % de completitud, ya que no se identificaron valores faltantes en ninguna de sus variables.
* Geolocation: todas las variables presentan un 100 % de completitud.
* Orders: se identificaron valores faltantes en las columnas order_delivered_customer_date (2,98 %), order_delivered_carrier_date (1,79 %) y order_approved_at (0,16 %). Estos valores pueden corresponder a pedidos que aún no habían sido aprobados o entregados al momento del registro, por lo que no necesariamente representan errores en los datos.
* Order Items: todas las variables presentan un 100 % de completitud.
* Order Payments: no se encontraron valores faltantes.
* Order Reviews: se observó un alto porcentaje de valores faltantes en review_comment_title (88,34 %) y review_comment_message (58,70 %). Esto es consistente con el hecho de que los comentarios textuales son opcionales y muchos clientes únicamente asignan una calificación sin escribir una reseña.
* roducts: se identificaron valores faltantes en product_category_name, product_name_lenght, product_description_lenght y product_photos_qty (1,85 % cada una), así como un porcentaje mínimo (0,01 %) en las variables de peso y dimensiones.
* Sellers: presenta un 100 % de completitud.
* Category Translation: no presenta valores faltantes.

En general, el conjunto de datos presenta un alto nivel de completitud. Los valores faltantes se concentran principalmente en las tablas order_reviews, debido a la naturaleza opcional de los comentarios de los clientes, y products, donde el porcentaje de datos faltantes es bajo. Estos casos serán tratados durante la fase de preparación de los datos.

##  Consistencia de los datos



### Distribución de los datos para establecer los supuestos de rango esperado


In [46]:
print(products.describe())

       product_name_lenght  product_description_lenght  product_photos_qty  \
count         32341.000000                32341.000000        32341.000000   
mean             48.476949                  771.495285            2.188986   
std              10.245741                  635.115225            1.736766   
min               5.000000                    4.000000            1.000000   
25%              42.000000                  339.000000            1.000000   
50%              51.000000                  595.000000            1.000000   
75%              57.000000                  972.000000            3.000000   
max              76.000000                 3992.000000           20.000000   

       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7.0

In [47]:
print(order_items.describe())

       order_item_id          price  freight_value
count  112650.000000  112650.000000  112650.000000
mean        1.197834     120.653739      19.990320
std         0.705124     183.633928      15.806405
min         1.000000       0.850000       0.000000
25%         1.000000      39.900000      13.080000
50%         1.000000      74.990000      16.260000
75%         1.000000     134.900000      21.150000
max        21.000000    6735.000000     409.680000


In [48]:
print(order_payments.describe())

       payment_sequential  payment_installments  payment_value
count       103886.000000         103886.000000  103886.000000
mean             1.092679              2.853349     154.100380
std              0.706584              2.687051     217.494064
min              1.000000              0.000000       0.000000
25%              1.000000              1.000000      56.790000
50%              1.000000              1.000000     100.000000
75%              1.000000              4.000000     171.837500
max             29.000000             24.000000   13664.080000


In [49]:
print(order_reviews.describe())

       review_score
count  99224.000000
mean       4.086421
std        1.347579
min        1.000000
25%        4.000000
50%        5.000000
75%        5.000000
max        5.000000


In [50]:
print(geolocation.describe())

       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng
count                 1.000163e+06     1.000163e+06     1.000163e+06
mean                  3.657417e+04    -2.117615e+01    -4.639054e+01
std                   3.054934e+04     5.715866e+00     4.269748e+00
min                   1.001000e+03    -3.660537e+01    -1.014668e+02
25%                   1.107500e+04    -2.360355e+01    -4.857317e+01
50%                   2.653000e+04    -2.291938e+01    -4.663788e+01
75%                   6.350400e+04    -1.997962e+01    -4.376771e+01
max                   9.999000e+04     4.506593e+01     1.211054e+02


### **Supuestos de rango esperado**

| Variable               | Estadísticos observados   | Supuesto de consistencia                                                                                                                                                             | Resultado esperado                                                                                                            |
| ---------------------- | ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------- |
| `price`                | Min = 0.85                | El precio debe ser **mayor o igual a 0**.                                                                                                                                            | No deberían existir precios negativos.                                                                                        |
| `freight_value`        | Min = 0.00                | El costo de envío debe ser **mayor o igual a 0**.                                                                                                                                    | Un valor de 0 puede corresponder a envíos gratuitos.                                                                          |
| `payment_installments` | Min = 0                   | El número de cuotas debe ser un **entero mayor o igual a 0**. Los registros con valor 0 serán revisados para determinar si corresponden a tipos de pago donde las cuotas no aplican. | No deberían existir valores negativos.                                                                                        |
| `payment_value`        | Min = 0                   | El valor del pago debe ser **mayor o igual a 0**.                                                                                                                                    | Los valores iguales a 0 serán revisados para confirmar si corresponden a pagos con cupones, descuentos u otros casos válidos. |
| `review_score`         | Min = 1, Max = 5          | La calificación debe estar entre **1 y 5**.                                                                                                                                          | Todos los registros deberían cumplir esta regla.                                                                              |
| `product_weight_g`     | Min = 0                   | El peso del producto debe ser **mayor que 0**. Los registros con peso igual a 0 serán revisados para determinar si corresponden a errores de captura o información faltante.         | No deberían existir pesos negativos; los valores 0 serán evaluados.                                                           |
| `product_length_cm`    | Min = 7                   | La longitud del producto debe ser **mayor que 0**.                                                                                                                                   | No se esperan inconsistencias.                                                                                                |
| `product_height_cm`    | Min = 2                   | La altura del producto debe ser **mayor que 0**.                                                                                                                                     | No se esperan inconsistencias.                                                                                                |
| `product_width_cm`     | Min = 6                   | El ancho del producto debe ser **mayor que 0**.                                                                                                                                      | No se esperan inconsistencias.                                                                                                |
| `product_photos_qty`   | Min = 1                   | La cantidad de fotografías debe ser **mayor o igual a 1**.                                                                                                                           | No se esperan inconsistencias.                                                                                                |
| `geolocation_lat`      | Min = -36.6, Max = 45.1   | La latitud debe estar entre **-90 y 90** grados.                                                                                                                                     | Todos los registros deberían cumplir esta regla.                                                                              |
| `geolocation_lng`      | Min = -101.4, Max = 121.1 | La longitud debe estar entre **-180 y 180** grados.                                                                                                                                  | Todos los registros deberían cumplir esta regla.                                                                              |


### Valores fuera del rango esperado

In [44]:
print(customers.columns)
print(geolocation.columns)
print(orders.columns)
print(order_items.columns)
print(order_payments.columns)
print(order_reviews.columns)
print(products.columns)
print(sellers.columns)
print(category_translation.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['product_id', 'prod

#### Comprobar inconsistencias

In [51]:
# Precios negativos
order_items[order_items["price"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [52]:
order_items[order_items["freight_value"] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [53]:
order_payments[order_payments["payment_installments"] < 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [54]:
order_payments[order_payments["payment_value"] < 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [55]:
order_reviews[
    (order_reviews["review_score"] < 1) |
    (order_reviews["review_score"] > 5)
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [58]:
products[products["product_weight_g"] <= 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


Se definió como supuesto que el peso de un producto debe ser mayor que 0 g. La validación identificó 4 registros con peso igual a 0 g. Aunque estos productos cuentan con dimensiones, categoría y descripción registradas, un peso nulo resulta poco consistente para un producto físico. Por ello, estos casos se consideran posibles inconsistencias y serán tratados durante la fase de preparación de los datos.

In [59]:
(products["product_length_cm"] <= 0).sum()

(products["product_height_cm"] <= 0).sum()

(products["product_width_cm"] <= 0).sum()

np.int64(0)

In [60]:
(products["product_photos_qty"] < 1).sum()

np.int64(0)

In [61]:
(
    (geolocation["geolocation_lat"] < -90) |
    (geolocation["geolocation_lat"] > 90)
).sum()

np.int64(0)

In [62]:
(
    (geolocation["geolocation_lng"] < -180) |
    (geolocation["geolocation_lng"] > 180)
).sum()

np.int64(0)

Se validaron las variables numéricas de acuerdo con los supuestos de consistencia definidos previamente. No se identificaron precios, costos de envío, valores de pago ni calificaciones fuera de los rangos esperados. Sin embargo, se encontraron 2 productos con peso igual a 0, los cuales serán revisados durante la etapa de limpieza para determinar si corresponden a errores de captura o a información faltante.

## Consistencia temporal

In [63]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

- Revisar que la aprobación no puede ser antes de la compra

In [64]:
orders[
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


- El envío no puede ocurrir antes de la aprobación

In [67]:
print(orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
])

                               order_id                       customer_id  \
15     dcb36b511fcac050b97cd5c05de84dc3  3b6828a50ffe546942b7a473d70ac0fc   
64     688052146432ef8253587b930b01a06d  81e08b08e5ed4472008030d70327c71f   
199    58d4c4747ee059eeeb865b349b41f53a  1755fad7863475346bc6c3773fe055d3   
210    412fccb2b44a99b36714bca3fef8ad7b  c6865c523687cb3f235aa599afef1710   
415    56a4ac10a4a8f2ba7693523bb439eede  78438ba6ace7d2cb023dbbc81b083562   
...                                 ...                               ...   
99091  240ead1a7284667e0ec71d01f80e4d5e  fcdd7556401aaa1c980f8b67a69f95dc   
99230  78008d03bd8ef7fcf1568728b316553c  043e3254e68daf7256bda1c9c03c2286   
99266  76a948cd55bf22799753720d4545dd2d  3f20a07b28aa252d0502fe7f7eb030a9   
99377  a6bd1f93b7ff72cc348ca07f38ec4bee  6d63fa86bd2f62908ad328325799152f   
99406  7fd85cb0143de098a4c5ab5a57bfbd91  d32034dfc685b1ae15dd4c78eace868e   

      order_status order_purchase_timestamp   order_approved_at  \
15      

Se identificaron 1359 pedidos en los que la fecha de entrega al transportista es anterior a la fecha de aprobación. Estos registros representan posibles inconsistencias temporales o desfases en el registro de eventos y deberán revisarse antes del análisis.

- La entrega no puede ser antes del envío

In [68]:
print(orders[
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
])

                               order_id                       customer_id  \
6437   a1abeb653a4d4cd1e142ccb8c82cd069  5f50465da00b7fed5dd1239f4ecf6e2c   
9553   383aa8b2724fe452d9ccd9934a8c628b  b1cb2f9d7a19480f3749e248db14d58f   
13487  cb1134f9010d242e9515ad1c78ec0c39  2fd33ac77677bd214b1882868317eeed   
14474  dceb62e8fa94b46006c9554fed743df0  2721900eb4e0f1cc2c836dd7bc1b1e11   
19268  5f9d46795c3126674e52becb3a1a517f  79287bcaafdde5c793b996fc40bb7d9f   
21338  8c78d01de3a9009e23d6877a7cc9be20  6cd7106899e59a1fbd0622d5f1efedf4   
22520  b27af682321527a6349f1761eb3f360c  9859dd92e872dbaa60ca3cd5f0d7ad07   
25393  1cc3ae63caffff2d6c3ee3e78e074acf  01c843a2c0600def0b7693dba47af460   
25646  e37f11cae9985ca58f0b56f268720537  3947a361301f2ff0f3223159a0f2701c   
27470  fa3e37584f4fdb1ded0e0de700dfcb4e  63be4feff10a0b1d85f2cfbf10df9754   
34939  c1e2bf2b7dd3309f2f5356c6b63968fa  e37d47e7eec62f08dc5deecc7d5532d6   
41636  b866af202be0692766081310cd4085e1  d1800078046ed2e5ae1b0792b695c56e   

Se encontraron 23 pedidos cuya fecha de entrega al cliente es anterior a la fecha en que el transportista recibió el pedido. Estos registros constituyen inconsistencias temporales y deberán ser revisados durante la etapa de limpieza.

## Verificación de consistencia en variables categóricas

In [69]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"\n{'='*20} {nombre} {'='*20}")

    # Seleccionar variables categóricas (tipo object)
    columnas_cat = df.select_dtypes(include="object").columns

    for col in columnas_cat:
        print(f"\nVariable: {col}")
        print(f"Número de categorías: {df[col].nunique(dropna=False)}")
        print(df[col].value_counts(dropna=False).head(10))


==================== Customers ====================

Variable: customer_id
Número de categorías: 99441
customer_id
274fa6071e5e17fe303b9748641082c8    1
e5ed7280cd1a3ac2ba29fd6650d8867c    1
c6ece8a5137f3c9c3a3a12302a19a2ac    1
821a7275a08f32975caceff2e08ea262    1
5eef6cce1f34954c9e7004332388ccc7    1
be631308cb609ff74d0e0fb54815e18c    1
a1b5ca506b592bb72d4caadcbfe71385    1
30c96385d694acb8aa2dc0df1770120b    1
b7c889215de76857c7967c1011125d2d    1
c156d63bdfce1d456bd43cf1c4dadfca    1
Name: count, dtype: int64

Variable: customer_unique_id
Número de categorías: 96096
customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
dc813062e0fc23409cd255f7f53c7074     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
Name: count

Se revisaron las variables categóricas de todas las tablas mediante el análisis del número de categorías y de las frecuencias de cada una. En general, las variables presentan un comportamiento consistente y no se identificaron valores inesperados en campos como order_status, payment_type, customer_state y seller_state, cuyos valores corresponden a las categorías esperadas del dominio.

Se evaluó la consistencia de las bases de datos mediante la revisión de registros duplicados, tipos de datos, valores fuera de rango, secuencia temporal de las fechas y coherencia de las variables categóricas. En general, las tablas presentan un buen nivel de consistencia, ya que la mayoría de los atributos cumplen con el tipo de dato esperado, no se identificaron duplicados en las tablas principales (excepto en geolocation, donde existen 261.831 registros duplicados), y los valores numéricos se encuentran dentro de los rangos definidos para cada variable. Asimismo, las variables categóricas mantienen categorías coherentes con el dominio del negocio, aunque se identificaron diferencias de formato en algunos valores textuales, como el uso de mayúsculas, minúsculas y acentos en nombres de ciudades y comentarios de las reseñas. Adicionalmente, la validación temporal permitió detectar registros cuya secuencia de fechas no sigue el orden esperado del proceso logístico, los cuales serán revisados durante la etapa de limpieza antes de la integración de los datos.

## Trazabilidad de los datos